# Đánh giá 4 model YOLO (window detection) trên tập TEST — vUDT Sub-module 4

Notebook này **chỉ đánh giá**, không train — dùng 4 file weight (`yolov8s`, `yolo11s`, `yolov8m`, `yolo11m`) đã train từ 2 session trước.

**Nội dung**:
1. Tính metrics trên tập `test`: Precision, Recall, AP50, AP75, mAP50:95, Mean IoU (TP), thời gian detect trung bình
2. Vẽ bounding box lên toàn bộ ảnh test cho từng model: **box đỏ = model detect (kèm điểm confidence)**, **box xanh lá = ground-truth (nhãn chuẩn)**. Đường vẽ mỏng, chữ nhỏ vì mật độ cửa sổ/ảnh khá cao (~50 box/ảnh)
3. Nén toàn bộ (CSV metrics + 4 thư mục ảnh bbox) thành 1 file zip

In [1]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.124 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7035.6/8062.4 GB disk)


In [ ]:
!ls /kaggle/input/

In [5]:
import os

# ============================================================
# ⚠️ SỬA 2 DÒNG DƯỚI cho đúng với tên dataset đã mount
KAGGLE_DATA_ROOT = "/kaggle/input/datasets/chungquc/dataset-window/data_Train"        # <-- chứa images/test, labels/test
KAGGLE_WEIGHTS_ROOT = "/kaggle/input/datasets/chungquc/model-yolo-window/model"            # <-- chứa 4 file .pt
# ============================================================

WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

assert os.path.isdir(KAGGLE_DATA_ROOT), f"Không tìm thấy: {KAGGLE_DATA_ROOT}"
assert os.path.isdir(KAGGLE_WEIGHTS_ROOT), f"Không tìm thấy: {KAGGLE_WEIGHTS_ROOT}"

print("Nội dung KAGGLE_WEIGHTS_ROOT:")
!ls "{KAGGLE_WEIGHTS_ROOT}"

for sub in ["images/test", "labels/test"]:
    p = os.path.join(KAGGLE_DATA_ROOT, sub)
    n = len(os.listdir(p)) if os.path.isdir(p) else -1
    print(f"{sub:15s} -> {n} files" if n >= 0 else f"{sub:15s} -> KHÔNG TỒN TẠI")

Nội dung KAGGLE_WEIGHTS_ROOT:
yolov11m_best.pt  yolov11s_best.pt  yolov8m_best.pt  yolov8s_best.pt
images/test     -> 122 files
labels/test     -> 122 files


In [6]:
# ============================================================
# ⚠️ SỬA nếu tên file .pt của bạn khác — khớp đúng tên file thật trong KAGGLE_WEIGHTS_ROOT
MODEL_WEIGHTS = {
    "yolov8s": os.path.join(KAGGLE_WEIGHTS_ROOT, "yolov8s_best.pt"),
    "yolo11s": os.path.join(KAGGLE_WEIGHTS_ROOT, "yolov11s_best.pt"),
    "yolov8m": os.path.join(KAGGLE_WEIGHTS_ROOT, "yolov8m_best.pt"),
    "yolo11m": os.path.join(KAGGLE_WEIGHTS_ROOT, "yolov11m_best.pt"),
}
# ============================================================

for name, path in MODEL_WEIGHTS.items():
    print(f"{name:10s} -> {'OK ✅' if os.path.exists(path) else 'THIẾU ❌'}  ({path})")

yolov8s    -> OK ✅  (/kaggle/input/datasets/chungquc/model-yolo-window/model/yolov8s_best.pt)
yolo11s    -> OK ✅  (/kaggle/input/datasets/chungquc/model-yolo-window/model/yolov11s_best.pt)
yolov8m    -> OK ✅  (/kaggle/input/datasets/chungquc/model-yolo-window/model/yolov8m_best.pt)
yolo11m    -> OK ✅  (/kaggle/input/datasets/chungquc/model-yolo-window/model/yolov11m_best.pt)


In [7]:
# Viết lại dataset.yaml với đường dẫn Kaggle (chỉ cần cho model.val())
yaml_content = f"""path: {KAGGLE_DATA_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: window
"""

yaml_path = os.path.join(WORK_DIR, "dataset.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(yaml_content)

path: /kaggle/input/datasets/chungquc/dataset-window/data_Train
train: images/train
val: images/val
test: images/test

names:
  0: window



## 1. Metrics trên tập TEST

- **Precision, Recall, AP50, AP75, mAP50:95** — lấy trực tiếp từ `model.val()` (vì single-class nên AP50=mAP50, AP75=mAP75)
- **Mean IoU (TP)** — IoU trung bình của các detection khớp đúng (IoU≥0.5 với ground-truth), Ultralytics không xuất sẵn số này nên tính thủ công bằng `box_iou` + greedy matching
- **Avg detect time (ms/ảnh)** — đo qua `model.predict()`

In [8]:
import numpy as np
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou


def yolo_to_xyxy(label_path, img_w, img_h):
    """Đọc file nhãn YOLO (.txt) và chuyển sang toạ độ pixel xyxy."""
    boxes = []
    if not os.path.exists(label_path):
        return np.zeros((0, 4), dtype=np.float32)
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            _, xc, yc, w, h = map(float, parts[:5])
            x1 = (xc - w / 2) * img_w
            y1 = (yc - h / 2) * img_h
            x2 = (xc + w / 2) * img_w
            y2 = (yc + h / 2) * img_h
            boxes.append([x1, y1, x2, y2])
    return np.array(boxes, dtype=np.float32)


def eval_mean_iou_and_speed(model, img_dir, lbl_dir, imgsz=1024, conf=0.25, iou_match_thres=0.5):
    """Ghép prediction <-> ground-truth bằng greedy matching theo IoU, trả về mean IoU của
    các cặp TP (IoU >= iou_match_thres) và thời gian inference trung bình mỗi ảnh (ms)."""
    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)
    img_files = sorted([p for p in img_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])

    matched_ious = []
    infer_times_ms = []

    for img_path in img_files:
        result = model.predict(source=str(img_path), imgsz=imgsz, conf=conf, verbose=False)[0]
        infer_times_ms.append(result.speed["inference"])

        with Image.open(img_path) as im:
            img_w, img_h = im.size

        gt_boxes = yolo_to_xyxy(lbl_dir / (img_path.stem + ".txt"), img_w, img_h)
        pred_boxes = result.boxes.xyxy.cpu().numpy() if len(result.boxes) else np.zeros((0, 4), dtype=np.float32)

        if len(gt_boxes) == 0 or len(pred_boxes) == 0:
            continue

        iou_matrix = box_iou(torch.tensor(pred_boxes), torch.tensor(gt_boxes)).numpy()

        used_pred, used_gt = set(), set()
        pairs = sorted(
            ((iou_matrix[p, g], p, g) for p in range(iou_matrix.shape[0]) for g in range(iou_matrix.shape[1])),
            key=lambda x: x[0], reverse=True,
        )
        for iou_val, p, g in pairs:
            if iou_val < iou_match_thres:
                break
            if p in used_pred or g in used_gt:
                continue
            used_pred.add(p)
            used_gt.add(g)
            matched_ious.append(iou_val)

    mean_iou = float(np.mean(matched_ious)) if matched_ious else 0.0
    avg_time_ms = float(np.mean(infer_times_ms)) if infer_times_ms else 0.0
    return mean_iou, avg_time_ms

In [9]:
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
DEVICE = 0 if NUM_GPUS >= 1 else "cpu"
print("Device:", DEVICE, "| GPU count:", NUM_GPUS)

test_img_dir = os.path.join(KAGGLE_DATA_ROOT, "images/test")
test_lbl_dir = os.path.join(KAGGLE_DATA_ROOT, "labels/test")

final_metrics = {}

for model_name, weight_path in MODEL_WEIGHTS.items():
    if not os.path.exists(weight_path):
        print(f"Bỏ qua {model_name}: thiếu file weight")
        continue

    print(f"\n{'='*60}\nĐánh giá {model_name}\n{'='*60}")
    model = YOLO(weight_path)

    val_metrics = model.val(
        data=yaml_path,
        split="test",
        imgsz=1024,
        device=DEVICE,
        project=f"{WORK_DIR}/runs_test",
        name=model_name,
        exist_ok=True,
        plots=True,
    )

    mean_iou, avg_time_ms = eval_mean_iou_and_speed(model, test_img_dir, test_lbl_dir, imgsz=1024)

    final_metrics[model_name] = {
        "Precision": val_metrics.box.mp,
        "Recall": val_metrics.box.mr,
        "AP50": val_metrics.box.map50,
        "AP75": val_metrics.box.map75,
        "mAP50:95": val_metrics.box.map,
        "Mean IoU (TP)": mean_iou,
        "Avg detect time (ms/img)": avg_time_ms,
    }

df = pd.DataFrame(final_metrics).T
df.to_csv(f"{WORK_DIR}/test_eval_summary.csv")
df

Device: 0 | GPU count: 2

Đánh giá yolov8s
Ultralytics 8.4.124 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.2 ms, read: 63.5±76.2 MB/s, size: 1052.5 KB)
val: Scanning /kaggle/input/datasets/chungquc/dataset-window/data_Train/labels/test... 122 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 122/122 268.1it/s 0.5s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/chungquc/dataset-window/data_Train/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.6s/it 13.2s0.6ss
                   all        122       6336      0.893      0.786      0.827      0.629
Speed: 5.8ms preprocess, 19.7ms inference, 0.0ms loss, 29.7ms postprocess per image
Results saved to /kaggle/working/runs_test/yolov8s

Đánh giá yolo11s
Ultralytics 8.4.124 🚀 Python-3.12.

,Precision,Recall,AP50,AP75,mAP50:95,Mean IoU (TP),Avg detect time (ms/img)
yolov8s,0.892633,0.785981,0.827373,0.739037,0.628920,0.875094,28.059468
yolo11s,0.896582,0.814394,0.857190,0.750526,0.647082,0.874666,23.733643
yolov8m,0.897216,0.794937,0.843031,0.760003,0.648268,0.880351,44.526487
yolo11m,0.895583,0.799400,0.847542,0.760425,0.650276,0.881221,42.460476


## 2. Vẽ bounding box trên tập TEST

- **Box đỏ**: dự đoán của model, kèm điểm confidence (không ghi tên nhãn vì chỉ có 1 class)
- **Box xanh lá**: ground-truth (nhãn chuẩn)
- Đường vẽ mỏng (thickness=1), chữ nhỏ — vì mật độ cửa sổ/ảnh khá cao, tránh rối ảnh

In [19]:
import cv2

BBOX_DIR = f"{WORK_DIR}/test_predictions"
os.makedirs(BBOX_DIR, exist_ok=True)

RED = (0, 0, 255)     # BGR - box model detect
GREEN = (0, 255, 0)   # BGR - box ground-truth
BOX_THICKNESS = 2      # tăng từ 1 -> 2 cho đậm hơn
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.4       # tăng nhẹ từ 0.35 -> 0.4
FONT_THICKNESS = 1
CONF_THRES = 0.25


def draw_gt_and_pred(img_path, gt_boxes, pred_boxes, pred_confs, out_path):
    img = cv2.imread(str(img_path))
    if img is None:
        return

    for (x1, y1, x2, y2) in gt_boxes:
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), GREEN, BOX_THICKNESS)

    for (x1, y1, x2, y2), conf in zip(pred_boxes, pred_confs):
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        cv2.rectangle(img, (x1, y1), (x2, y2), RED, BOX_THICKNESS)
        label = f"{conf:.2f}"
        text_y = y1 - 4 if y1 - 4 > 5 else y1 + 12
        cv2.putText(img, label, (x1, text_y), FONT, FONT_SCALE, RED, FONT_THICKNESS, cv2.LINE_AA)

    cv2.imwrite(str(out_path), img)

In [20]:
test_img_files = sorted([p for p in Path(test_img_dir).glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])

for model_name, weight_path in MODEL_WEIGHTS.items():
    if not os.path.exists(weight_path):
        print(f"Bỏ qua {model_name}: thiếu file weight")
        continue

    print(f"Đang vẽ bbox cho {model_name} ({len(test_img_files)} ảnh)...")
    model = YOLO(weight_path)

    out_dir = Path(BBOX_DIR) / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    for img_path in test_img_files:
        with Image.open(img_path) as im:
            img_w, img_h = im.size
        gt_boxes = yolo_to_xyxy(Path(test_lbl_dir) / (img_path.stem + ".txt"), img_w, img_h)

        result = model.predict(source=str(img_path), imgsz=1024, conf=CONF_THRES, verbose=False)[0]
        pred_boxes = result.boxes.xyxy.cpu().numpy() if len(result.boxes) else np.zeros((0, 4), dtype=np.float32)
        pred_confs = result.boxes.conf.cpu().numpy() if len(result.boxes) else np.zeros((0,), dtype=np.float32)

        draw_gt_and_pred(img_path, gt_boxes, pred_boxes, pred_confs, out_dir / img_path.name)

    print(f"  -> đã lưu vào {out_dir}")

Đang vẽ bbox cho yolov8s (122 ảnh)...
  -> đã lưu vào /kaggle/working/test_predictions/yolov8s
Đang vẽ bbox cho yolo11s (122 ảnh)...
  -> đã lưu vào /kaggle/working/test_predictions/yolo11s
Đang vẽ bbox cho yolov8m (122 ảnh)...
  -> đã lưu vào /kaggle/working/test_predictions/yolov8m
Đang vẽ bbox cho yolo11m (122 ảnh)...
  -> đã lưu vào /kaggle/working/test_predictions/yolo11m


## 3. Nén kết quả (metrics CSV + 4 thư mục ảnh bbox)

In [21]:
import shutil

export_dir = f"{WORK_DIR}/export_eval"
os.makedirs(export_dir, exist_ok=True)

shutil.copy(f"{WORK_DIR}/test_eval_summary.csv", f"{export_dir}/test_eval_summary.csv")

bbox_export = os.path.join(export_dir, "bbox_predictions")
os.makedirs(bbox_export, exist_ok=True)
for model_name in MODEL_WEIGHTS:
    src = os.path.join(BBOX_DIR, model_name)
    dst = os.path.join(bbox_export, model_name)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        print(f"Thiếu {src}, bỏ qua")

zip_path = f"{WORK_DIR}/vUDT_window_eval_results"
shutil.make_archive(zip_path, "zip", export_dir)
print(f"Đã nén xong: {zip_path}.zip")

Đã nén xong: /kaggle/working/vUDT_window_eval_results.zip
